# ap2111_analyze_data

## Reading an Organization ID and Analyzing the Corresponding Data

### Summary

This notebook reads an organization ID (`organization_id`) and retrieves the corresponding data either via the **data_loading.py module** or from a **CSV export**.  
The dataset is then subjected to comprehensive quality, plausibility, and consistency checks. Subsequent analyses are conducted for metering points, generators, and consumers, including visual assessments.

The code can be used as a starting point for analyses of **Renewable Energy Communities (RECs)**. In this notebook, **REC 12** is used.  

**Special attention must be given to specific characteristics of the data, such as gaps in time series, anomalies in values, and other irregularities, which should be addressed separately during the analysis.**


### Notebook Structure

1. **Imports and Definitions**

    - Import of all required Python libraries and definition of global constants, parameters, and helper functions used throughout the notebook.

2. **Data Loading**

    - Loading of raw data from the available data sources and preparation of the initial dataset for analysis.

3. **Exploratory Data Analysis**

    - **3.1 Feature Overview**
        
    - **3.2 Descriptive Statistics**
        
    - **3.3 Validation of Numerical Features**
        
        - **3.3.1 Plausibility Checks** 
        
        - **3.3.2 Unique Values per Feature** 
        
        - **3.3.3 Daylight Saving Time Consistency Check**
        
        - **3.3.4 Distribution of Weighted Measured Generation**
        
        - **3.3.5 Distribution of Weighted Measured Consumption**

4. **Advanced Data Analysis**

    - **4.1 Analysis of Number of Generators and Consumers**
        
    - **4.2 Unique Metering Points per Day Over Time** 
        
    - **4.3 Temporal Analysis of Generator and Consumer Behavior**
        
        - **4.3.1 Single-Day Example for an Individual Metering Point**

        - **4.3.2 Hourly Surplus Generation on Selected Days**

        - **4.3.3 Hourly Consumption and Surplus Generation**

        - **4.3.4 Hourly Generation and Surplus Energy for a Selected Metering Point**

        - **4.3.5 Hourly Consumption and Community Coverage for a Selected Metering Point**

        - **4.3.6 Daily Energy Consumption Over Time**

        - **4.3.7 Daily Energy Generation Over Time**

        - **4.3.8 Daily Energy Consumption for a Selected Metering Point**

    - **4.4 Aggregation of Numerical Features for Generation, Consumption and Surplus** 

5. **Time Series Gap Analysis**

    - Identification of missing timestamps and gaps within the time series data.

6. **Analysis of Aggregated Data per REC**

    - Analysis of datasets aggregated at REC level.



### **1. Imports and Definitions**  

In [ ]:
# standard libs
import sys
from datetime import datetime
from pathlib import Path
import os

# data & viz
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import streamlit as st

# env & db
from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder

# project path
PROJECT_ROOT = Path.cwd().resolve().parents[1]  # repo root, assuming this notebook runs from src/notebooks/
sys.path.append(str(PROJECT_ROOT / "src"))

# project modules
from modules.data_loading import (load_energy_community_data, load_all_metering_points_in_energy_community_data, get_postgres_engine,)


### **2. Data Loading**

This section defines the core parameters required for data ingestion:
- Selection of the data source (**CSV** or **database**)
- Specification of the **REC organization_id**
- Definition of the **time range** for data extraction
- Configuration of the **path to local CSV files**, if applicable

In [ ]:
use_csv = 0  # 1 = load data from CSV files, 0 = load data from database

# database configuration
org_id = 2  # ID of the Renewable Energy Community (REC)
time_start = datetime(2025, 1, 1)  # define the start date as datetime(year, month, day)
time_end = datetime(2025, 9, 30)   # define the end date as datetime(year, month, day)

# CSV configuration (only needed if using CSV instead of database)
path_to_local_data = "../../local_data/"  # relative path to local CSV data directory
org_csv = "example.csv" 

# runtime metadata
date = datetime.now()  # timestamp of script execution

In [ ]:
if use_csv == 0:

    # load database credentials from .env file
    load_dotenv()

    ssh_host = os.getenv("SSH_HOST")
    ssh_port = int(os.getenv("SSH_PORT"))
    ssh_user = os.getenv("SSH_USER")
    ssh_pw = os.getenv("SSH_PASSWORD")

    postgres_server_ip = os.getenv("POSTGRES_SERVER_IP")
    postgres_port = int(os.getenv("POSTGRES_PORT"))

    # establish SSH tunnel to securely access the remote PostgreSQL database
    with SSHTunnelForwarder(
        (ssh_host, ssh_port),
        ssh_username=ssh_user,
        ssh_password=ssh_pw,
        remote_bind_address=(ssh_host, postgres_port),
        local_bind_address=(postgres_server_ip, postgres_port)
    ) as tunnel:
        df=load_all_metering_points_in_energy_community_data(org_id=org_id, time_start=time_start, time_end=time_end, sql_engine=get_postgres_engine())


In [ ]:
if use_csv == 1:
    # load input data from local CSV file instead of database
    df = pd.read_csv(path_to_local_data + org_csv)

In [ ]:
df.sample(5) 

In [ ]:
# copy dataframe and convert time column to datetime
org_df = df.copy()
org_df["time"] = pd.to_datetime(org_df["time"])  # make datetime column

In [ ]:
print(org_df["time"].dtype)

### **3. Exploratory Data Analysis**

#### 3.1 Feature Overview

The feature descriptions provided below are based on the contents of beschreibung.md and have been made easier to understand through illustrative examples.

In [ ]:
org_df.sample(5) 

- **organization_id**: ID of the respective Renewable Energy Community (REC).

- **metering_point_id**: ID of a metering point. Each metering point represents a single energy direction. For example, a household can have one metering point as either a consumer or a producer. In cases where it both consumes and produces energy (a prosumer), it may have two or even multiple metering points.

- **time**: Timestamp of the energy flow, including timezone information.

- **period_interval**: Duration of the energy flow interval (15 minutes). Data is recorded every quarter hour.

- **energy_direction**: Direction of the energy flow (C = Consumption, G = Generation).

- **wt_meas_cons**: Weighted measured consumption in kWh, adjusted by the participation factor. This represents the amount of energy actually consumed by a consumer considering its share in the REC. “Weighted” means that the potential participation factor is already accounted for. For example, if the participation factor is 0.5, `wt_meas_cons` represents half of the household’s actual consumption.

- **comm_pot**: Community potential in kWh. This is the amount of energy produced within the REC to a consumer available. If this value is higher than `comm_cov`, it indicates that the consumer did not fully utilize the energy allocated to it from the REC.

- **comm_cov**: Community coverage in kWh, representing consumed energy produced within the REC. This is the amount of energy that the REC can provide. If `comm_cov` equals `comm_pot`, the consumer has fully utilized the energy potential allocated to them by the REC.

- **wt_meas_gen**: Weighted measured generation in kWh, adjusted by the participation factor. This represents the amount of energy generated by a producer for the REC. If `wt_surp_gen` is 0, it means that all generated energy was consumed within the REC and none was fed into the grid. “Weighted” indicates that the participation factor is already applied. For example, if the factor is 0.5, `wt_meas_gen` represents half of the household’s actual generation.

- **wt_surp_gen**: Surplus generation in kWh, representing the portion of `wt_meas_gen` that exceeds current demand. If this value is greater than zero, it indicates that the REC is producing more energy than required, with the surplus fed into the grid. For example, if `wt_meas_gen` is 2.5 kWh and `wt_surp_gen` is 1 kWh, this means that 1.5 kWh of the generated energy is consumed within the REC.


#### 3.2 Descriptive Statistics

This table provides a summary of all features in the dataframe, showing their data types, the number of missing values, and their minimum and maximum values.

*Note:* Missing values in these numerical features are expected and normal. For example, if a metering point corresponds to a consumer, it would not make sense for it to produce any energy (`wt_meas_gen`).

- `wt_meas_cons`  
- `comm_pot`  
- `comm_cov`  
- `wt_meas_gen`  
- `wt_surp_gen`

In [ ]:
# Summary statistics of the dataframe
summary = pd.DataFrame({
    "Data Type": org_df.dtypes,
    "Missing Values": org_df.isna().sum(),
    "Min": org_df.min(numeric_only=False),
    "Max": org_df.max(numeric_only=False)
})

summary = summary.reset_index().rename(columns={"index": "Feature"})

print(summary)

If the total number of entries in the dataframe is equal to the sum of missing values in `wt_meas_cons` and `wt_meas_gen`, it can be assumed that all other entries in the dataset are complete.

In [ ]:
# Total number of entries in the dataframe
total_entries = len(org_df)

# Number of missing values for wt_meas_cons and wt_meas_gen
missing_values = org_df[['wt_meas_cons', 'wt_meas_gen']].isna().sum().sum()

print(f"Total entries in the dataframe: {total_entries}")
print(f"Total missing values in wt_meas_cons and wt_meas_gen: {missing_values}")

missing_ratio = missing_values / total_entries
print(f"Proportion of missing values: {missing_ratio:.2%}")

#### 3.3 Validation of Numerical Features

This section provides a validation of the numerical features, a Daylight Saving Time consistency check, and an assessment of the general plausibility of the data.


##### 3.3.1 Plausibility Checks

For the numerical features `"wt_meas_cons"`, `"comm_pot"`, `"comm_cov"`, `"wt_meas_gen"`, and `"wt_surp_gen"`:

- Values should not be below 0.  
- Extremely high values are possible, but values above 100 are unlikely, except in cases involving large commercial consumers, which should not be included in this dataset.

In [ ]:
# Check if values are between 0 and 100 for selected columns
cols = ["wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]
all_ok = True

for col in cols:
    # ignore missing values
    invalid_values = df.loc[~df[col].between(0, 100), col].dropna()
    if not invalid_values.empty:
        print(f"Warning: Values outside 0-100 found in '{col}':")
        print(invalid_values)
        all_ok = False

if all_ok:
    print("All values in the selected columns are between 0 and 100!")

##### 3.3.2 Unique Values per Feature

The following check describes key categorical features in the dataset and their expected values:

- **metering_point_id**: The number of metering points can vary depending on the REC.

- **period_interval**: 1 unique value: `0 days 00:15:00`  
  This represents the standardized data recording interval (15 minutes) and should always be consistent.

- **energy_direction**: 2 unique values: `['C', 'G']`  
  These indicate the direction of energy flow (Consumption or Generation) and are fixed by definition.


In [ ]:
# Display unique values for selected columns
cols = ["metering_point_id", "period_interval", "energy_direction"]

for col in cols:
    unique_vals = org_df[col].unique()
    print(f"\n{col} — {len(unique_vals)} unique values:")
    print(unique_vals)

##### 3.3.3 Daylight Saving Time Consistency Check

The following section performs a validation of the time series to identify potential issues related to Daylight Saving Time (DST). Specifically, during the autumn transition, the clock is set back by one hour, which may result in **duplicate time entries** in the dataset.

Regardless of whether DST-related issues occur, duplicates may still be present. These must be identified and resolved before the data is used for modeling.  

In general, data can still be retrospectively corrected by the respective RECs within three months of its creation. Therefore, it is recommended **not to include the last three months before the current date** in analysis and modeling.

In [ ]:
# Check for duplicate time entries per metering point
check_time = org_df.groupby(["metering_point_id", "time"]).size()
duplicates = check_time[check_time > 1]

if duplicates.empty:
    print("No duplicate time entries found!")
else:
    print("Duplicate time entries found:")
    print(duplicates)

In [ ]:
# Check for duplicate time entries per metering point
duplicates = org_df.groupby(["metering_point_id", "time"]).size()
duplicates = duplicates[duplicates > 1]

# Extract unique dates with duplicates
duplicate_dates = sorted(set(duplicates.index.get_level_values("time").date))

# Print dates in a readable format
print("Dates with duplicates:")
for date in duplicate_dates:
    print(f"- {date}")

##### 3.3.4 Distribution of Weighted Measured Generation

Most recorded generation values (`wt_meas_gen`) should be relatively small, typically below 1 kWh per interval. This suggests that the dataset is dominated by small-scale producers such as photovoltaic (PV) systems, whose output varies with the time of day and season, while larger generators, such as small hydropower plants, are not represented.  

It should also be noted that `wt_meas_gen` only represents the **surplus generation** within a member of the REC. Therefore, if a member produces a large amount of energy but consumes most of it immediately, `wt_meas_gen` can still appear low.


In [ ]:
plt.figure(figsize=(10, 4))  # set figure size

# Create histogram with 50 bins
org_df["wt_meas_gen"].hist(bins=50)

plt.title("Distribution of Weighted Measured Generation")  # plot title # Sample plot: distribution of 'wt_meas_gen'
plt.xlabel("Weighted Measured Generation [kWh]")  # x-axis label
plt.ylabel("Frequency [number of observations]")  # y-axis label

plt.grid(True)
plt.show()


##### 3.3.5 Distribution of Weighted Measured Consumption

The distribution of `wt_meas_con` (Weighted Measured Consumption) should be similar to the distribution of `wt_meas_gen`. Larger consumption values per interval could indicate the presence of a large consumer within the REC.  

However, it should be noted that `wt_meas_cons` aswell does **not** provide insight into the internal distribution of consumption within a single consumer.

In [ ]:
plt.figure(figsize=(10, 4))  # set figure size

# Create histogram with 50 bins
org_df["wt_meas_cons"].hist(bins=50)

plt.title("Distribution of Weighted Measured Consumption")  # plot title # Sample plot: distribution of 'wt_meas_cons'
plt.xlabel("Weighted Measured Consumption [kWh]")  # x-axis label
plt.ylabel("Frequency [number of observations]")  # y-axis label

plt.grid(True)
plt.show()

### **4. Advanced Data Analysis**

In [ ]:
# Create a copy of the original dataframe for EEG cleaning to preserve the raw data
eeg_cleaned = org_df.copy()
eeg_cleaned.head(5)

#### 4.1 Analysis of Number of Generators and Consumers

The size of an REC can be inferred from the number of members. However, the dataset does not indicate which metering points belong to the same member, for example, a single-family household. 

The fact that there may be more consumers than generators does not necessarily imply a general supply shortage within the community.

In [ ]:
#  Count the number of metering points by energy direction
consumer = eeg_cleaned[eeg_cleaned["energy_direction"] == "C"]
generators = eeg_cleaned[eeg_cleaned["energy_direction"] == "G"]

# Unique metering points overall and by type
total_mp = eeg_cleaned["metering_point_id"].nunique()
total_consumers = consumer["metering_point_id"].nunique()
total_generators = generators["metering_point_id"].nunique()

# Print summary with percentages
print(f"Total metering points in EEG {org_id}: {total_mp}")
print(f"\tConsumers: {total_consumers} ({total_consumers / total_mp * 100:.1f}%)")
print(f"\tGenerators: {total_generators} ({total_generators / total_mp * 100:.1f}%)")

#### 4.2 Unique Metering Points per Day Over Time

The chart below probably reveals that the number of metering points varies over the observation period. Both the number of consumers and generators increases and decreases over time, although it remains unclear whether a participant simultaneously acts as both a consumer and generator. 

In [ ]:
# Calculation of the number of unique counting points per day
eeg_cleaned["time_day"] = pd.to_datetime(eeg_cleaned["time"]).dt.floor("D")

# Count the number of unique org_ids for each individual day.
daily_counts = (
    eeg_cleaned.groupby(["time_day", "energy_direction"])["metering_point_id"]
    .nunique()
    .reset_index(name="unique_obj_count")
)

# If energy_direction is not set, it is replaced with 0.
daily_counts = (
    daily_counts
    .pivot(index="time_day", columns="energy_direction", values="unique_obj_count")
    .fillna(0)
    .reset_index()
)

In [ ]:
# Plotly chart of count of metering points per day
fig = px.area(
    daily_counts,
    x="time_day",
    y=["C", "G"],  # Consumer + Generator
    title="Number of Unique Metering Points per Day Over Time",
    labels={
        "time_day": "Time",
        "value": "Number of Unique Metering Points per Day",
        "variable": "Energy Direction",
    },
    color_discrete_map={
        "C": "#1f77b4",  # blue for consumer
        "G": "#ff7f0e"   # orange for generator
    }
)

fig.update_layout(
    hovermode="x unified",
    legend_title_text="Energy Direction",
    template="plotly_white"
)

fig.show()

#### 4.3 Temporal Analysis of Generator and Consumer Behavior

In this chapter, we will examine the behavior of individual metering points in more detail.

##### 4.3.1 Single-Day Example for an Individual Metering Point

This section allows the selection of a specific metering point and a time period to filter the dataset. The filtered entries display all measurements for the chosen metering point within the specified start and end timestamps.  

This section can be used, for example, to inspect duplicates in detail if you already know the time period and the metering point where they occurred.

By examining the data below, it is possible to observe for example how much energy a metering point generates at night or how energy consumption increases during the morning hours.  
However, it is not possible from this data alone to determine whether variations are due to weather conditions on that day (e.g., cloud cover) or changes in the energy consumption within the entity.

In [ ]:
# Choose metering point
metering_point_id = 16263
df_single = org_df[org_df["metering_point_id"] == metering_point_id]

# Choose start and end of the period (make them timezone-aware to match df)
start_datetime = pd.to_datetime("2025-07-01 05:45", utc=True)
end_datetime   = pd.to_datetime("2025-07-01 13:00", utc=True)

df_filtered = df_single[
    (df_single["time"] >= start_datetime) &
    (df_single["time"] <= end_datetime)
]

df_filtered

##### 4.3.2 Hourly Surplus Generation on Selected Days

This figure displays the hourly surplus energy (`wt_surp_gen`) for selected days. Users can choose a specific time period to filter the data, allowing analysis of how surplus generation evolves throughout the day and across different dates.

The observed patterns may vary significantly depending on the specific REC and the chosen dates. Surplus energy occurs often between 5:00 a.m. and 4:00 p.m., peaking around midday, consistent with PV system generation. Afternoon declines may be due to system orientation (e.g., east-facing panels). Variations between days also reflect participant consumption and weather conditions, such as lower surplus on 1 August 2025 possibly indicating cloudiness or higher demand.

In [ ]:
# Choose specific days for analysis
selected_dates = [
    pd.to_datetime("2025-07-01").date(),
    pd.to_datetime("2025-08-01").date(),
    pd.to_datetime("2025-09-01").date(),
]

# Use the original dataframe
df = org_df.copy()

# Filter for the selected dates
df = df[df["time"].dt.date.isin(selected_dates)]

# Keep only rows with surplus generation values
df = df[df["wt_surp_gen"].notna()]

# Extract hour and date for grouping
df["hour"] = df["time"].dt.hour
df["date"] = df["time"].dt.date

# Aggregate surplus generation hourly
df_hourly = (
    df.groupby(["date", "hour"])["wt_surp_gen"]
    .sum()
    .reset_index(name="surplus_kWh")
)

# Define custom colors for each date
color_map = {
    pd.to_datetime("2025-07-01").date(): "#1f77b4",  # blue
    pd.to_datetime("2025-08-01").date(): "#ff7f0e",  # orange
    pd.to_datetime("2025-09-01").date(): "#2ca02c",  # green
}

# Create line plot of hourly surplus generation
fig = px.line(
    df_hourly,
    x="hour",
    y="surplus_kWh",
    color="date",
    markers=True,
    title="Hourly Surplus Generation on Selected Days",
    labels={
        "hour": "Hour of Day",
        "surplus_kWh": "Surplus Energy [kWh]",
        "date": "Date",
    },
    color_discrete_map=color_map
)

fig.update_layout(
    xaxis=dict(dtick=1),
    template="plotly_white",
    hovermode="x unified",
)

fig.show()

##### 4.3.3 Hourly Consumption and Surplus Generation

This figures display the hourly consumption (`wt_meas_cons`) and surplus generation (`wt_surp_gen`) on selected days. Users can choose a specific time period to filter the data, allowing analysis of how surplus generation evolves throughout the day and across different dates.

The observed patterns may vary significantly depending on the specific REC and the chosen dates. The blue curves probably indicate that hourly consumption is generally low during the day and higher in the early morning and evening, while surplus energy occurs during periods of low demand, reflecting either reduced consumption, increased generation, or both.

In [ ]:

# Choose specific days for analysis
selected_days = [
    pd.to_datetime("2025-07-01").date(),
    pd.to_datetime("2025-08-01").date(),
    pd.to_datetime("2025-09-01").date(),
]

# Use the original dataframe
df = org_df.copy()

# Extract hour and date for grouping
df["hour"] = df["time"].dt.hour
df["date"] = df["time"].dt.date

# Aggregate consumption and surplus generation hourly
df_hourly = (
    df.groupby(["date", "hour"])
    .agg(
        consumption_kWh=("wt_meas_cons", "sum"),
        surplus_kWh=("wt_surp_gen", "sum"),
    )
    .reset_index()
)

# Transform to long format for plotting multiple metrics
df_long = df_hourly.melt(
    id_vars=["date", "hour"],
    value_vars=["consumption_kWh", "surplus_kWh"],
    var_name="Metric",
    value_name="Energy [kWh]"
)

# Rename metrics for clearer labels
df_long["Metric"] = df_long["Metric"].replace({
    "consumption_kWh": "Consumption [kWh]",
    "surplus_kWh": "Surplus Energy [kWh]"
})

# Define colors for each metric
color_map = {
    "Consumption [kWh]": "#1f77b4",      # blue
    "Surplus Energy [kWh]": "#ff7f0e",   # orange
}

# Plot hourly consumption and surplus generation for each selected day
for day in selected_days:
    df_day = df_long[df_long["date"] == day]  # Filter for the current day

    fig = px.line(
        df_day,
        x="hour",
        y="Energy [kWh]",
        color="Metric",
        line_dash="Metric",
        markers=True,
        title=f"Hourly Consumption and Surplus Generation on {day}",
        labels={
            "hour": "Hour of Day",
            "Energy [kWh]": "Energy [kWh]",
            "Metric": "Energy Type",
        },
        color_discrete_map=color_map
    )

    # Customize layout
    fig.update_layout(
        xaxis=dict(dtick=1),
        template="plotly_white",
        hovermode="x unified",
        legend_title_text="Energy Metric",
    )

    fig.show()


##### 4.3.4 Hourly Generation and Surplus Energy for a Selected Metering Point

The following figure shows hourly generation (`wt_meas_gen`) and surplus energy (`wt_surp_gen`) for a selected metering point on a specific day.  

The chart below shows the energy generated by a selected metering point on a given day. The dotted line represents surplus energy, indicating when the generated energy exceeds local consumption. This pattern probably reflects typical PV-system behavior, with peak generation occurring during periods of lower overall demand within the community.

In [ ]:
# Use the original dataframe
df = org_df.copy()

# Select a specific metering point first
metering_point_id = 15985 
df_single = df[df["metering_point_id"] == metering_point_id]

# Choose specific days for analysis
selected_days = ["2025-07-01"]
df_single = df_single[df_single["time"].dt.date.astype(str).isin(selected_days)]

# Extract hour and date for grouping
df_single["hour"] = df_single["time"].dt.hour
df_single["date"] = df_single["time"].dt.date

# Aggregate generation and surplus energy hourly
df_hourly = (
    df_single.groupby(["date", "hour"])[["wt_meas_gen", "wt_surp_gen"]]
    .sum()
    .reset_index()
)

# Transform to long format for plotting multiple metrics
df_long = df_hourly.melt(
    id_vars=["date", "hour"],
    value_vars=["wt_meas_gen", "wt_surp_gen"],
    var_name="Energy Type",
    value_name="Energy [kWh]"
)

# Rename metrics for clearer labels
df_long["Energy Type"] = df_long["Energy Type"].replace({
    "wt_meas_gen": "Generation [kWh]",
    "wt_surp_gen": "Surplus Energy [kWh]"
})

# Define colors for each metric
color_map = {
    "Generation [kWh]": "green",
    "Surplus Energy [kWh]": "orange",
}

# Create line plot of hourly generation and surplus energy
fig = px.line(
    df_long,
    x="hour",
    y="Energy [kWh]",
    color="Energy Type",
    line_dash="Energy Type",
    markers=True,
    title=f"Hourly Generation and Surplus Energy for Metering Point {metering_point_id} on {selected_days[0]}",
    labels={"hour": "Hour of Day", "Energy [kWh]": "Energy [kWh]", "Energy Type": "Metric"},
    color_discrete_map=color_map
)

# Customize layout
fig.update_layout(
    xaxis=dict(dtick=1),
    template="plotly_white",
    hovermode="x unified",
)

fig.show()

##### 4.3.5 Hourly Consumption and Community Coverage for a Selected Metering Point

The following figure shows hourly consumption (`wt_meas_con`) and community coverage (`comm_cov`) for a selected metering point on a specific day.  

The chart below shows a selected metering point on a given day. Energy demand is generally higher during the early morning, evening, and night, while it is lower during daytime. This pattern typically reflects the presence of on-site PV generation, which covers most of the energy demand during daylight hours, resulting in surplus supply primarily occurring outside of peak generation times, as indicated by the surplus energy line in the figure.

In [ ]:
# Use the original dataframe
df = org_df.copy()

# Choose a metering point
metering_point_id = 15606
df_single = df[df["metering_point_id"] == metering_point_id]

# Choose a date
selected_days = ["2025-07-01"]
df_single = df_single[df_single["time"].dt.date.astype(str).isin(selected_days)]

df_single["hour"] = df_single["time"].dt.hour
df_single["date"] = df_single["time"].dt.date

df_hourly = (
    df_single.groupby(["date", "hour"])[["wt_meas_cons", "comm_cov"]]
    .sum()
    .reset_index()
)

df_long = df_hourly.melt(
    id_vars=["date", "hour"],
    value_vars=["wt_meas_cons", "comm_cov"],
    var_name="Energy Type",
    value_name="Energy [kWh]"
)

df_long["Energy Type"] = df_long["Energy Type"].replace({
    "wt_meas_cons": "Consumption [kWh]",
    "comm_cov": "Community Coverage [kWh]"
})

color_map = {
    "Consumption [kWh]": "green",
    "Community Coverage [kWh]": "orange",

}

# Create line plot of hourly generation and surplus energy
fig = px.line(
    df_long,
    x="hour",
    y="Energy [kWh]",
    color="Energy Type",
    line_dash="Energy Type",
    markers=True,
    title=f"Hourly Consumption and Community Coverage for Metering Point {metering_point_id} on {selected_days[0]}",
    labels={"hour": "Hour of Day", "Energy [kWh]": "Energy [kWh]", "Energy Type": "Metric"},
    color_discrete_map=color_map
)

# Customize layout
fig.update_layout(
    xaxis=dict(dtick=1),
    template="plotly_white",
    hovermode="x unified",
)

fig.show()

##### 4.3.6 Daily Energy Consumption Over Time

This line chart shows the energy consumption (`wt_meas_cons`) over time.

The chart below likely suggests that energy consumption increases over time. However, this trend is most probably driven by the growing number of members within the REC or by seasonal changes in user behavior. Therefore, this visualization does not provide a particularly high level of informational value.


In [ ]:
# Tagesaggregation
daily = consumer.resample("D", on="time")["wt_meas_cons"].sum()

plt.figure(figsize=(12, 6))
plt.plot(daily.index, daily.values, linewidth=2)

plt.title("Daily Energy Consumption Over Time", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Consumption [kWh]")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

plt.show()



##### 4.3.7 Daily Energy Generation Over Time

This line chart shows the energy consumption (`wt_meas_gen`) over time. 

The chart below is likely dependent on the number of members, similar to the previous chart, and may also reflect seasonal effects. In particular, for RECs that generate a large share of their energy from photovoltaic systems, significantly higher energy production is expected during the summer months.

In [ ]:
# Tagesaggregation
daily = generators.resample("D", on="time")["wt_meas_gen"].sum()

plt.figure(figsize=(12, 6))
plt.plot(daily.index, daily.values, linewidth=2)

plt.title("Daily Energy Generation Over Time", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Generation [kWh]")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

plt.show()

##### 4.3.8 Daily Energy Consumption for a Selected Metering Point

These visualizations show the daily energy consumption (`wt_meas_cons`) over time for a specific metering points.

The charts should indicate that each metering point exhibits a specific usage pattern. Theoretically, one should be able to observe seasonal dependencies, and over the long term, the usage behavior should remain relatively stable.

In [ ]:
metering_points = [16263, 15606, 15963]

for id in metering_points:
    df_consumer = consumer[
        consumer['metering_point_id'] == id
    ].sort_values('time')

    fig = px.line(
        df_consumer,
        x="time",
        y="wt_meas_cons",
        title=f"wt_meas_cons over time for metering_point_id {id}"
    )

    fig.show()

#### 4.4 Aggregation of Numerical Features for Generation, Consumption and Surplus

In [ ]:
sum_eeg_cons = consumer["wt_meas_cons"].sum()
sum_eeg_gen = generators["wt_meas_gen"].sum()
sum_eeg_surp = generators["wt_surp_gen"].sum()
print(f"Sum wt_meas_cons of all metering points: {sum_eeg_cons:.0f} kWh")
print(f"Sum wt_meas_gen of all metering points: {sum_eeg_gen:.0f} kWh")
print(f"Sum wt_surp_gen of all metering points: {sum_eeg_surp:.0f} kWh")

### **5. Time Series Gap Analysis**

The first part of the code checks for gaps and irregular time intervals for each metering point individually, recording the maximum gap and whether the intervals deviate from the expected 15-minute spacing. The second part performs a global check across the entire dataset to identify any missing timestamps in the full 15-minute time range.



In [ ]:
# Final list for gaps over REC
gaps = []

# group for mp_id
for mp_id, group in eeg_cleaned.groupby("metering_point_id"):
    group = group.sort_values("time")

    # Time differences between consecutive points in time
    diffs = group["time"].diff().dropna()

    # max gap
    max_gap = diffs.max()

    # If gaps are not constant 
    if len(diffs.unique()) > 1:
        gaps.append({
            "mp_id": mp_id,
            "number_of_missing_values": len(group),
            "max_gap": max_gap,
            "deviating_distances": True
        })
    else:
        gaps.append({
            "mp_id": mp_id,
            "number_of_missing_values": len(group),
            "max_gap": max_gap,
            "deviating_distances": False
        })

# safe as df
gap_df = pd.DataFrame(gaps)
gap_df = gap_df.sort_values(by="number_of_missing_values", ascending=False)

if len(gap_df) == 0:
    print("No missing time entries found!")
else:   
    print(gap_df)

In [ ]:
# check missings over eeg
full_range = pd.date_range(
    start=eeg_cleaned["time"].min(),
    end=eeg_cleaned["time"].max(),
    freq="15T"
)

missing_times = full_range.difference(eeg_cleaned["time"])

if len(missing_times) == 0:
    print("No missing time entries found!")
else:   
    print(missing_times)

### **6. Analysis of Aggregated Data per REC**

The following code is used to visualize the aggregated data per REC for the features: `wt_meas_cons`, `comm_pot`, `comm_cov`, `wt_meas_gen`, and `wt_surp_gen`.


In [ ]:
cols_to_mean = ["wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]
df_agg = (
    eeg_cleaned.groupby("time")[cols_to_mean]
      .mean()
      .reset_index()
)

In [ ]:
df_agg.head(-1)

In [ ]:
summary = pd.DataFrame({
        'dtype': df_agg.dtypes,
        'missing': df_agg.isna().sum(),
        'min': df_agg.min(numeric_only=False),
        'max': df_agg.max(numeric_only=False)
    })

print(summary)

In [ ]:
# plots
for c in cols_to_mean:

    fig = px.line(
        df_agg,
        x="time",
        y=c,
        title=f"{c} over time",
    )

    fig.show()


### Findings

- Community potential is sometimes very high, over 100 kWh per interval; this may indicate many PV systems (see Chapter 3.3.1).

- The REC is large, with more than 914 metering points (see Chapter 4.2).

- No transition between summer and winter time was detected; no adjustments are required. However, many duplicate values exist (see Chapters 3.3.3 and 4.3.1).

- All measured values lie within acceptable and expected ranges.

- All values appear plausible; no erroneous or extreme values were observed.

